## Phase 1: Library Imports and Data Cleaning
Load the dataset and transform the list of strings into a One-Hot matrix for the Apriori algorithm.

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings('ignore')

CLUSTER_NAMES_MAP = {
    0: "The Plant-Based Household",
    1: "The Family Budget Optimizer",
    2: "The Promotion-Driven Pet Parent",
    3: "The Demanding Premium Tech Consumer",
    4: "The Wellness Customer",
    5: "The Affluent Health-Conscious Buyer"
}


def load_and_prepare_data(basket_path='customer_basket (1).csv', clusters_path='dataset_clusters.csv'):
    print("Loading data files...")
    customer_basket = pd.read_csv(basket_path, index_col='invoice_id')
    customer_clusters = pd.read_csv(clusters_path)
    
    if customer_clusters.index.name == 'customer_id':
        customer_clusters = customer_clusters.reset_index()
    elif 'customer_id' not in customer_clusters.columns and customer_clusters.columns[0] == 'Unnamed: 0':
        customer_clusters = customer_clusters.rename(columns={'Unnamed: 0': 'customer_id'})
    elif 'customer_id' not in customer_clusters.columns:
        customer_clusters = customer_clusters.reset_index().rename(columns={'index': 'customer_id'})

    for col in ['cluster', 'Cluster']:
        if col in customer_clusters.columns:
            customer_clusters[col] = customer_clusters[col].map(CLUSTER_NAMES_MAP)
            customer_clusters = customer_clusters.rename(columns={col: 'cluster'})
    
    merged_df = customer_clusters.merge(customer_basket, how='inner', on='customer_id')
    return merged_df


def convert_string_to_list(string):
    return json.loads(string.replace("'", '"'))


def calculate_metric_range(rules_df, metric='lift'):
    if rules_df.empty:
        return f"{metric} Range: N/A (No rules generated)"
    min_val = rules_df[metric].min()
    max_val = rules_df[metric].max()
    return f"{metric} Range: {max_val - min_val:.4f} (Min: {min_val:.4f}, Max: {max_val:.4f})"


def plot_top_products(basket_list, title_suffix=""):
    te = TransactionEncoder()
    te_ary = te.fit(basket_list).transform(basket_list)
    df_trans = pd.DataFrame(te_ary, columns=te.columns_)
    
    top_items = df_trans.sum().sort_values(ascending=False).head(10)
    
    plt.figure(figsize=(10, 5))
    top_items.plot(kind='bar', color='#5c6bc0', edgecolor='black')
    plt.title(f'Top 10 Most Frequent Products - {title_suffix}', fontweight='bold')
    plt.ylabel('Absolute Frequency')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    filename = f"top_products_{title_suffix.lower().replace(' ', '_')}.png"
    plt.savefig(filename)
    plt.close()


def plot_association_results(rules_df, title_suffix=""):
    if rules_df.empty:
        return
    plt.figure(figsize=(8, 5))
    scatter = plt.scatter(rules_df['support'], rules_df['confidence'], 
                          c=rules_df['lift'], cmap='viridis', alpha=0.6)
    plt.colorbar(scatter, label='Lift')
    plt.title(f'Rules Scatter Plot (Support vs Confidence) - {title_suffix}', fontweight='bold')
    plt.xlabel('Support')
    plt.ylabel('Confidence')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    filename = f"rules_scatter_{title_suffix.lower().replace(' ', '_')}.png"
    plt.savefig(filename)
    plt.close()


def run_association_rules(basket_data, min_support=0.05, min_confidence=0.2, split_train=True):
    if split_train:
        train_size = int(len(basket_data) * 0.8)
        data_to_model = basket_data[:train_size]
    else:
        data_to_model = basket_data
        
    te = TransactionEncoder()
    te_fit = te.fit(data_to_model).transform(data_to_model)
    transactions_items = pd.DataFrame(te_fit, columns=te.columns_)
    
    frequent_itemsets = apriori(transactions_items, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        return frequent_itemsets, pd.DataFrame()
        
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
    rules = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)
    return frequent_itemsets, rules


if __name__ == "__main__":
    try:
        df_basket_clusters = load_and_prepare_data(
            basket_path='customer_basket (1).csv', 
            clusters_path='dataset_clusters.csv'
        )
        df_basket_clusters['list_of_goods_parsed'] = df_basket_clusters['list_of_goods'].apply(convert_string_to_list)
        
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        pd.set_option('display.max_colwidth', None)

        def clean_rules_df(df_rules):
            if df_rules.empty:
                return df_rules
            df_cool = df_rules.copy()
            df_cool['antecedents'] = df_cool['antecedents'].apply(lambda x: f"[{', '.join(list(x))}]")
            df_cool['consequents'] = df_cool['consequents'].apply(lambda x: f"[{', '.join(list(x))}]")
            metrics = ['support', 'confidence', 'lift', 'leverage', 'conviction', 'zhangs_metric']
            for m in metrics:
                if m in df_cool.columns:
                    df_cool[m] = df_cool[m].round(4)
            return df_cool

        def clean_items_df(df_items):
            if df_items.empty:
                return df_items
            df_cool = df_items.copy()
            df_cool['itemsets'] = df_cool['itemsets'].apply(lambda x: f"[{', '.join(list(x))}]")
            df_cool['support'] = df_cool['support'].round(4)
            return df_cool

        # =========================================================================
        # PART 1: GLOBAL RULES (ALL CUSTOMERS)
        # =========================================================================
        print("\n" + "="*80)
        print("=== PART 1: GLOBAL ASSOCIATION RULES (ALL CUSTOMERS COMBINED) ===")
        print("="*80)
        all_baskets = df_basket_clusters['list_of_goods_parsed'].tolist()
        plot_top_products(all_baskets, "Global Dataset")
        
        items_all, rules_all = run_association_rules(all_baskets, min_support=0.02, min_confidence=0.15, split_train=True)
        
        print(f"\n-> [Global] Frequent Itemsets (Top 10):")
        display(clean_items_df(items_all.head(10)))
        
        print(f"\n-> [Global] Association Rules Generated (Top 15 sorted by Lift):")
        if not rules_all.empty:
            df_all_view = rules_all[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15)
            display(clean_rules_df(df_all_view))
            print(f"\n{calculate_metric_range(rules_all, 'lift')}")
            plot_association_results(rules_all, "Global Dataset")
        else:
            print("No rules generated globally for the current threshold limits.")

        # =========================================================================
        # PART 2: STANDARDIZED ANALYSIS PER CLUSTER (AUTOMATED LOOP)
        # =========================================================================
        print("\n" + "="*80)
        print("=== PART 2: STANDARDIZED ASSOCIATION RULES ANALYSIS BY CLUSTER COEFFICIENT ===")
        print("="*80)
        
        DEFAULT_SUPPORT = 0.02
        DEFAULT_CONFIDENCE = 0.15
        
        for cluster_id, cluster_name in CLUSTER_NAMES_MAP.items():
            print("\n" + "-"*60)
            print(f"ANALYSIS FOR CLUSTER {cluster_id}: {cluster_name}")
            print("-"*60)
            
            df_sub_cluster = df_basket_clusters[df_basket_clusters['cluster'] == cluster_name]
            cluster_baskets = df_sub_cluster['list_of_goods_parsed'].tolist()
            
            print(f"Total transactions within this segment: {len(cluster_baskets)}")
            
            if len(cluster_baskets) > 0:
                plot_top_products(cluster_baskets, cluster_name)
                
                items_cl, rules_cl = run_association_rules(
                    cluster_baskets, 
                    min_support=DEFAULT_SUPPORT, 
                    min_confidence=DEFAULT_CONFIDENCE, 
                    split_train=True
                )
                
                print(f"\n   -> Frequent Itemsets Found (Top 10 Sample out of {len(items_cl)}):")
                if not items_cl.empty:
                    display(clean_items_df(items_cl.head(10)))
                else:
                    print(f"      No frequent itemsets passed the {DEFAULT_SUPPORT*100}% support threshold.")
                
                print(f"\n   -> Association Rules Found (Top 10 sorted by Lift out of {len(rules_cl)}):")
                if not rules_cl.empty:
                    df_cl_view = rules_cl[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)
                    display(clean_rules_df(df_cl_view))
                    print(f"\n      {calculate_metric_range(rules_cl, 'lift')}")
                    plot_association_results(rules_cl, cluster_name)
                else:
                    print(f"      No rules generated for the thresholds (Support={DEFAULT_SUPPORT*100}%, Confidence={DEFAULT_CONFIDENCE*100}%).\n"
                          f"      Indicates independent purchasing behavior for products within this segment.")
            else:
                print(f"   -> Alert: No records found in the dataset for this cluster.")

        print("\n" + "="*80)
        print("=== DATA PROCESSING COMPLETED SUCCESSFULLY ===")
        print("="*80)

    except Exception as e:
        print(f"\n[Execution Error]: {e}")

Loading data files...

=== PART 1: GLOBAL ASSOCIATION RULES (ALL CUSTOMERS COMBINED) ===

-> [Global] Frequent Itemsets (Top 10):


,support,itemsets
0,0.1190,[airpods]
1,0.0633,[almonds]
2,0.0695,[antioxydant juice]
3,0.1275,[asparagus]
4,0.0666,[avocado]
5,0.0837,[babies food]
6,0.0686,[bacon]
7,0.0511,[barbecue sauce]
8,0.0552,[beer]
9,0.0449,[black beer]



-> [Global] Association Rules Generated (Top 15 sorted by Lift):


,antecedents,consequents,support,confidence,lift
0,[airpods],[energy drink],0.0245,0.2060,2.8112
1,[energy drink],[airpods],0.0245,0.3346,2.8112
2,[airpods],[bluetooth headphones],0.0237,0.1992,2.8008
3,[bluetooth headphones],[airpods],0.0237,0.3334,2.8008
4,[cereals],[eggs],0.0226,0.2241,2.4039
5,[eggs],[cereals],0.0226,0.2430,2.4039
6,[butter],[eggs],0.0209,0.2143,2.2993
7,[eggs],[butter],0.0209,0.2238,2.2993
8,[cereals],[fresh bread],0.0228,0.2256,2.2803
9,[fresh bread],[cereals],0.0228,0.2305,2.2803



lift Range: 0.7249 (Min: 2.0863, Max: 2.8112)

=== PART 2: STANDARDIZED ASSOCIATION RULES ANALYSIS BY CLUSTER COEFFICIENT ===

------------------------------------------------------------
ANALYSIS FOR CLUSTER 0: The Plant-Based Household
------------------------------------------------------------
Total transactions within this segment: 7492

   -> Frequent Itemsets Found (Top 10 Sample out of 173):


...


      lift Range: 0.8936 (Min: 2.0119, Max: 2.9055)

------------------------------------------------------------
ANALYSIS FOR CLUSTER 5: The Affluent Health-Conscious Buyer
------------------------------------------------------------
Total transactions within this segment: 3131

   -> Frequent Itemsets Found (Top 10 Sample out of 177):


,support,itemsets
0,0.1150,[airpods]
1,0.0607,[almonds]
2,0.0635,[antioxydant juice]
3,0.1238,[asparagus]
4,0.0703,[avocado]
5,0.0847,[babies food]
6,0.0627,[bacon]
7,0.0583,[barbecue sauce]
8,0.0523,[beer]
9,0.0459,[black beer]



   -> Association Rules Found (Top 10 sorted by Lift out of 26):


,antecedents,consequents,support,confidence,lift
0,[tooth brush],[shampoo],0.0216,0.2967,4.2454
1,[shampoo],[tooth brush],0.0216,0.3086,4.2454
2,[shower gel],[shampoo],0.0220,0.2696,3.8577
3,[shampoo],[shower gel],0.0220,0.3143,3.8577
4,[shower gel],[tooth brush],0.0204,0.2500,3.4396
5,[tooth brush],[shower gel],0.0204,0.2802,3.4396
6,[cereals],[fresh bread],0.0260,0.2600,2.6573
7,[fresh bread],[cereals],0.0260,0.2653,2.6573
8,[eggs],[cereals],0.0248,0.2594,2.5983
9,[cereals],[eggs],0.0248,0.2480,2.5983



      lift Range: 2.0962 (Min: 2.1492, Max: 4.2454)

=== DATA PROCESSING COMPLETED SUCCESSFULLY ===


## Main Marketing Insights & Actions

### Insight A: Global Store Drivers (Traffic Magnets)
* **Finding:** Across the entire store, items like `asparagus`, `airpods`, and `cereals` dominate absolute purchasing frequencies. 
* **Marketing Action:** Treat these as **"Loss-Leaders"** in mass-media campaigns or catalog covers. High-reach discounts on these specific items will efficiently drive overall traffic to the checkout.

### Insight B: Cluster 4 — "The Wellness Customer" (High-Conversion Target)
* **Finding:** This segment yielded the most robust patterns, generating **14 strong association rules** with a **Maximum Lift of 2.87**. Customers are nearly 3 times more likely to buy a secondary healthy item if promoted next to the cluster's anchor item (`asparagus`).
* **Marketing Action — "Live Well" Campaign:** Launch immediate product bundles or targeted discount coupons pairing `asparagus` with its linked wellness items. Trigger automated "Frequently Bought Together" carousels at the digital checkout for this cluster.

### Insight C: Cluster 3 — "The Demanding Premium Tech Consumer" (Anti-Bundling Strategy)
* **Finding:** Cluster 3 generated **no meaningful association rules**. Purchases within this segment are statistically independent (buying Tech Product A does not influence buying Tech Product B).
* **Marketing Action — Individual Margin Strategy:** **Do not waste budget** creating product bundles or multi-pack discounts for this group. Pivot exclusively to **Up-Selling** (premium upgrades of the same item) or brand-loyalty perks (e.g., Apple- or Samsung-specific campaigns).
